# EyeAI AMD3 — Corrected Binary Training Pipeline

This notebook consumes the prepared dataset only. Data preparation is not repeated here.

Available runs:

- Run 05: corrected HYAMD-only EfficientNetV2-S baseline.
- Run 06: HYAMD + source-aware ARMD Curated EfficientNetV2-S training.
- Run 07: HYAMD-only fine-tuning from the best Run 06 checkpoint.
- Run 08: RETFound CFP with the last 6 Transformer blocks unfrozen.
- Run 09: RETFound CFP with the last 10 blocks unfrozen; run only if Run 08 underfits.
- Run 10: RETFound CFP with the last 12 blocks unfrozen; run only if Run 09 improves reliably.

Validation policy:

- HYAMD validation is the primary checkpoint-selection and threshold-tuning set.
- Held-out ARMD Curated positives are evaluated separately for external AMD recall.
- The locked HYAMD test manifest is never loaded by the training script.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

REPO_OWNER = "MozaicAI-Solutions"
REPO_NAME = "eyeai-team-AMD3"
BRANCH = "main"
REPO_DIR = Path("/kaggle/working/eyeai-team-AMD3")

# EfficientNet runs are disabled by default because Run 06 is already complete.
RUN_BASELINE = False
RUN_MIXED = False
RUN_FINETUNE = False
RUN_CROSS_VALIDATION = False

# Start with Run 08 only. Enable Run 09 or Run 10 after reviewing the previous run.
RUN_RETF_LAST6 = True
RUN_RETF_LAST10 = False
RUN_RETF_LAST12 = False

BASELINE_CONFIG = "configs/train_efficientnetv2_binary_run05_hyamd_corrected.yaml"
MIXED_CONFIG = "configs/train_efficientnetv2_binary_run06_mixed_external.yaml"
FINETUNE_CONFIG = "configs/train_efficientnetv2_binary_run07_hyamd_finetune.yaml"

RETF_CONFIG_FILES = {
    6: "configs/train_retfound_binary_run08_last6_mixed.yaml",
    10: "configs/train_retfound_binary_run09_last10_mixed.yaml",
    12: "configs/train_retfound_binary_run10_last12_mixed.yaml",
}


In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("GITHUB_TOKEN secret is missing.")

repo_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
print("Repository and package are ready.")


In [ ]:
def find_prepared_dataset_root():
    direct = Path("/kaggle/working/eyeai_prepared_binary_dataset")
    if (direct / "dataset_summary.json").exists():
        return direct

    matches = list(Path("/kaggle/input").rglob("dataset_summary.json"))
    valid = [
        path.parent
        for path in matches
        if (path.parent / "manifests" / "train_mixed.csv").exists()
    ]
    if len(valid) == 1:
        return valid[0]
    if not valid:
        raise FileNotFoundError("Prepared EyeAI dataset was not found under /kaggle/input.")
    raise RuntimeError(f"Multiple prepared datasets were found: {valid}")

DATASET_ROOT = find_prepared_dataset_root()
required_manifests = [
    DATASET_ROOT / "manifests" / "train_mixed.csv",
    DATASET_ROOT / "manifests" / "hyamd_val.csv",
    DATASET_ROOT / "manifests" / "armd_curated_val_positive.csv",
]
missing = [path for path in required_manifests if not path.exists()]
if missing:
    raise FileNotFoundError(
        "The prepared dataset is an older version and lacks the new external validation manifest. "
        f"Missing: {missing}. Re-run notebook 06 and save a new Kaggle Dataset version."
    )

os.environ["EYEAI_DATASET_ROOT"] = str(DATASET_ROOT)
print("Prepared dataset root:", DATASET_ROOT)
print(json.dumps(json.loads((DATASET_ROOT / "dataset_summary.json").read_text()), indent=2))


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU is not enabled. Activate a Kaggle GPU accelerator before training.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Hugging Face authentication is needed only for optional EfficientNet runs.
if RUN_BASELINE or RUN_MIXED or RUN_FINETUNE or RUN_CROSS_VALIDATION:
    hf_token = secrets.get_secret("HF_TOKEN")
    if not hf_token:
        raise RuntimeError("HF_TOKEN secret is missing.")
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

    from huggingface_hub import hf_hub_download

    weights_path = hf_hub_download(
        repo_id="timm/tf_efficientnetv2_s.in21k_ft_in1k",
        filename="model.safetensors",
        token=hf_token,
    )
    print("EfficientNetV2-S weights:", weights_path)
else:
    print("EfficientNet Hugging Face setup skipped.")


In [ ]:
import yaml

# Locate the official colour-fundus RETFound checkpoint and create temporary configs
# with the resolved Kaggle path. The repository configs remain unchanged.
retfound_candidates = sorted(Path("/kaggle/input").rglob("RETFound_mae_natureCFP.pth"))
if not retfound_candidates:
    raise FileNotFoundError(
        "RETFound_mae_natureCFP.pth was not found under /kaggle/input. "
        "Attach the RETFound checkpoint Kaggle Dataset."
    )
if len(retfound_candidates) > 1:
    print("Multiple RETFound checkpoints found; using the first one:")
    for candidate in retfound_candidates:
        print("-", candidate)

RETFOUND_CHECKPOINT = retfound_candidates[0]
print("RETFound CFP checkpoint:", RETFOUND_CHECKPOINT)

resolved_config_dir = Path("/kaggle/working/eyeai_resolved_configs")
resolved_config_dir.mkdir(parents=True, exist_ok=True)
RESOLVED_RETF_CONFIGS = {}

for blocks, relative_config in RETF_CONFIG_FILES.items():
    source_path = REPO_DIR / relative_config
    config = yaml.safe_load(source_path.read_text(encoding="utf-8"))
    config["model"]["retfound_checkpoint_path"] = str(RETFOUND_CHECKPOINT)
    destination = resolved_config_dir / source_path.name
    destination.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    RESOLVED_RETF_CONFIGS[blocks] = str(destination)
    print(f"Resolved last-{blocks} config:", destination)


## Optional completed EfficientNet runs

Leave these disabled unless you intentionally want to reproduce Runs 05–07.


In [ ]:
os.chdir(REPO_DIR)
if RUN_BASELINE:
    subprocess.run(
        ["python", "-u", "scripts/train_binary.py", "--config", BASELINE_CONFIG, "--data-root", str(DATASET_ROOT)],
        check=True,
    )


In [ ]:
os.chdir(REPO_DIR)
if RUN_MIXED:
    subprocess.run(
        ["python", "-u", "scripts/train_binary.py", "--config", MIXED_CONFIG, "--data-root", str(DATASET_ROOT)],
        check=True,
    )


In [ ]:
os.chdir(REPO_DIR)
if RUN_FINETUNE:
    subprocess.run(
        ["python", "-u", "scripts/train_binary.py", "--config", FINETUNE_CONFIG, "--data-root", str(DATASET_ROOT)],
        check=True,
    )


## RETFound controlled unfreezing sequence

Run 08 first. Do not enable all three runs together.

- Proceed to Run 09 only when Run 08 shows underfitting: low Train AP, low HYAMD Validation AP, and a small gap.
- Proceed to Run 10 only when Run 09 improves HYAMD AP/AUC reliably without a large train-validation gap.
- If Train AP is high but Validation AP is low, increasing the number of trainable blocks is not the correct response.


In [ ]:
os.chdir(REPO_DIR)
if RUN_RETF_LAST6:
    subprocess.run(
        [
            "python", "-u", "scripts/train_binary.py",
            "--config", RESOLVED_RETF_CONFIGS[6],
            "--data-root", str(DATASET_ROOT),
        ],
        check=True,
    )


In [ ]:
os.chdir(REPO_DIR)
if RUN_RETF_LAST10:
    subprocess.run(
        [
            "python", "-u", "scripts/train_binary.py",
            "--config", RESOLVED_RETF_CONFIGS[10],
            "--data-root", str(DATASET_ROOT),
        ],
        check=True,
    )


In [ ]:
os.chdir(REPO_DIR)
if RUN_RETF_LAST12:
    subprocess.run(
        [
            "python", "-u", "scripts/train_binary.py",
            "--config", RESOLVED_RETF_CONFIGS[12],
            "--data-root", str(DATASET_ROOT),
        ],
        check=True,
    )


## Optional 3-fold EfficientNet comparison

This remains disabled. It uses patient-disjoint HYAMD folds and external ARMD training images only.


In [ ]:
if RUN_CROSS_VALIDATION:
    for fold in range(3):
        print(f"Running mixed EfficientNet fold {fold}")
        subprocess.run(
            [
                "python", "-u", "scripts/train_binary.py",
                "--config", MIXED_CONFIG,
                "--data-root", str(DATASET_ROOT),
                "--fold", str(fold),
            ],
            check=True,
        )


In [ ]:
import pandas as pd

runs_root = Path("/kaggle/working/eyeai_binary_ensemble/runs")
summary_files = sorted(runs_root.rglob("*_summary.json"))
rows = []
for path in summary_files:
    summary = json.loads(path.read_text(encoding="utf-8"))
    metrics = summary.get("validation_metrics_at_threshold", {}) or {}
    external_metrics = summary.get("external_positive_validation_at_threshold", {}) or {}
    rows.append({
        "model": summary.get("model_name"),
        "best_epoch": summary.get("best_epoch"),
        "threshold": summary.get("best_threshold"),
        "average_precision": summary.get("best_selection_score"),
        "auc": metrics.get("auc"),
        "macro_f1": metrics.get("macro_f1"),
        "precision_amd": metrics.get("precision_amd"),
        "recall_amd": metrics.get("recall_amd"),
        "specificity": metrics.get("specificity"),
        "external_positive_recall": external_metrics.get("recall_amd"),
        "external_mean_probability": external_metrics.get("mean_probability"),
        "summary_path": str(path),
    })

if rows:
    display(pd.DataFrame(rows).sort_values("average_precision", ascending=False))
else:
    print("No completed run summaries were found.")
